<a href="https://colab.research.google.com/github/normala127/NLP_Yelp_Review_Project/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark
!apt-get install openjdk-17-jdk-headless -qq


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("project").getOrCreate()

print(spark)

In [6]:
df = spark.read.option("mode", "PERMISSIVE").json(r"/content/yelp_academic_dataset_business.json")
df.printSchema()
df.show()

root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-- DriveThru: string (nullable = true)
 |    |-- GoodForDancing: str

In [ ]:
test=df.select('attributes')
test.collect()[8]

In [5]:
df_review = spark.read.json(r"/content/drive/MyDrive/dataprj/yelp_academic_dataset_review.json")

In [ ]:
print((df_review.count(), len(df_review.columns)))

(6990280, 9)


In [7]:
df_review.createOrReplaceTempView("review")
df.createOrReplaceTempView("business")

In [8]:
joined_df = spark.sql("""
SELECT t1.*, t2.*
FROM review t1
LEFT JOIN business t2 ON t2.business_id = t1.business_id
""")
joined_df.show(5)

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+-------+-------------+-----------+--------------------+-----------+------------+-----+-----+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|             address|          attributes|         business_id|          categories|        city|               hours|is_open|     latitude|  longitude|                name|postal_code|review_count|stars|state|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+-------+-------------+-----------+--------------------+-----------+------

In [10]:
joined_df = joined_df.drop(df['business_id'])

In [11]:
joined_df.columns

['business_id',
 'cool',
 'date',
 'funny',
 'review_id',
 'stars',
 'text',
 'useful',
 'user_id',
 'address',
 'attributes',
 'categories',
 'city',
 'hours',
 'is_open',
 'latitude',
 'longitude',
 'name',
 'postal_code',
 'review_count',
 'stars',
 'state']

In [9]:
print((joined_df.count(), len(joined_df.columns)))

(6990280, 23)


In [26]:
df_checkin = spark.read.option("mode", "PERMISSIVE").json(r"/content/yelp_academic_dataset_checkin.json")

In [27]:
df_checkin.show()

+--------------------+--------------------+
|         business_id|                date|
+--------------------+--------------------+
|---kPU91CF4Lq2-Wl...|2020-03-13 21:10:...|
|--0iUa4sNDFiZFrAd...|2010-09-13 21:43:...|
|--30_8IhuyMHbSOcN...|2013-06-14 23:29:...|
|--7PUidqRWpRSpXeb...|2011-02-15 17:12:...|
|--7jw19RH9JKXgFoh...|2014-04-21 20:42:...|
|--8IbOsAAxjKRoYsB...|2015-06-06 01:03:...|
|--9osgUCSDUWUkoTL...|2015-06-13 02:00:...|
|--ARBQr1WMsTWiwOK...|2014-12-12 00:44:...|
|--FWWsIwxRwuw9vIM...|2010-09-11 16:28:...|
|--FcbSxK1AoEtEAxO...|2017-08-18 19:43:...|
|--LC8cIrALInl2vyo...|2017-01-12 19:10:...|
|--MbOh2O1pATkXa7x...|2013-04-21 01:52:...|
|--N9yp3ZWqQIm7DqK...|2012-10-06 20:46:...|
|--O3ip9NpXTKD4oBS...|2010-04-17 21:07:...|
|--OS_I7dnABrXvRCC...| 2018-05-11 18:23:36|
|--S43ruInmIsGrnnk...|2010-08-29 01:17:...|
|--SJXpAa0E-GCp2sm...|2014-04-06 22:23:...|
|--Sd93OFWITqDHifM...|2013-01-09 17:42:...|
|--ZVrH2X2QXBFdCil...|2010-08-12 18:21:...|
|--ZWv8kGlM2YL58uK...|2010-10-13

In [28]:
df_checkin.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)



In [31]:
df_checkin_new=df_checkin.withColumnRenamed('date', 'date_checkin')

In [32]:
df_checkin_new.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- date_checkin: string (nullable = true)



In [33]:
df_checkin_new.createOrReplaceTempView('checkin')

In [34]:
joined_df.createOrReplaceTempView('joined_df')

In [35]:
joined_df2 = spark.sql("""
SELECT t1.*, t2.date_checkin
FROM joined_df t1
JOIN checkin t2 ON t2.business_id = t1.business_id
""")
joined_df2.show(5)

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|         address|          attributes|          categories|           city|               hours|is_open|  latitude|  longitude|             name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|

In [36]:
joined_df2.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: 

In [42]:
df_isOpen=joined_df2.filter(joined_df2['is_open']==1)
df_isOpen.show()

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|         address|          attributes|          categories|           city|               hours|is_open|  latitude|  longitude|             name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|

In [39]:
df_isClosed=joined_df2.filter(joined_df2['is_open']==0)
df_isClosed.show()

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------+--------------------+--------------------+------------+--------------------+-------+----------+-----------+--------------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|       address|          attributes|          categories|        city|               hours|is_open|  latitude|  longitude|                name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------+--------------------+--------------------+------------+--------------------+-------+----------+-----------+--------------------+-----------+------------+-----+-----+--------------------+
|-0eUa8

In [43]:
print((df_isOpen.count(), len(df_isOpen.columns))) # 80% is open

(5616746, 23)


In [41]:
print((df_isClosed.count(), len(df_isClosed.columns))) # 20% is closed

(1180155, 23)


In [52]:
df_dropOpen=joined_df2.sampleBy('is_open', fractions={1:0.2, 0:1}, seed=42)

In [53]:
df_isClosed2=df_dropOpen.filter(df_dropOpen['is_open']==0)
df_isClosed2.show()

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------+--------------------+--------------------+------------+--------------------+-------+----------+-----------+--------------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|       address|          attributes|          categories|        city|               hours|is_open|  latitude|  longitude|                name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------+--------------------+--------------------+------------+--------------------+-------+----------+-----------+--------------------+-----------+------------+-----+-----+--------------------+
|-0eUa8

In [54]:
print((df_isClosed2.count(), len(df_isClosed2.columns))) # 80% is open

(1180155, 23)


In [55]:
df_isClosed3=df_dropOpen.filter(df_dropOpen['is_open']==1)
df_isClosed3.show()

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+--------------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|             address|          attributes|          categories|           city|               hours|is_open|  latitude|  longitude|                name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+--------------------+-----------+------------+-----+-----+--

In [56]:
print((df_isClosed3.count(), len(df_isClosed3.columns))) # 80% is open

(1122659, 23)


In [57]:
1122659-1180155

-57496

In [60]:
joined_df2.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- business_stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointm

In [66]:
cols=joined_df2.columns
stars_columns=[i for i, c in enumerate(cols) if c=='business_stars']

cols[stars_columns[0]]='stars_review'
cols[stars_columns[1]]='stars_business'

joined_df2=joined_df2.toDF(*cols)

In [67]:
joined_df2.columns

['business_id',
 'cool',
 'date',
 'funny',
 'review_id',
 'stars_review',
 'text',
 'useful',
 'user_id',
 'address',
 'attributes',
 'categories',
 'city',
 'hours',
 'is_open',
 'latitude',
 'longitude',
 'name',
 'postal_code',
 'review_count',
 'stars_business',
 'state',
 'date_checkin']

In [71]:
final_df=joined_df2.drop(*['cool', 'funny', 'useful', 'latitude', 'longitude', 'postal_code'])

In [72]:
final_df.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars_review: double (nullable = true)
 |-- text: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (n

In [76]:
df.write.mode("overwrite").json("/content/drive/MyDrive/dataprj/final.json")
